# Pico – large pre-training + grammar fine-tuning on Kaggle

**One-time setup**
1. Verify your phone number in Kaggle settings (needed for GPU + internet).
2. Upload your `train.jsonl` as a Kaggle Dataset (**Datasets → New Dataset**), then in this notebook **Add Input → your dataset**.
3. In the notebook sidebar, under **Session options**: set **Accelerator → GPU T4 x2** and turn **Internet → On**.

**Run it in the background:** click **Save Version → Save & Run All (Commit)**. It keeps running with your browser closed (up to 12 hours), then the results appear in the version's **Output** tab.

**If pre-training didn't finish in one session** (the log says *"Time limit reached"*): open the notebook, **Add Input → Notebook Output → this notebook's latest version**, and *Save & Run All* again. It picks up the data and checkpoint from the previous run and continues.

## 1. Setup

In [ ]:
import os, glob, shutil, time
START = time.time()          # Kaggle sessions end after 12 hours
W = '/kaggle/working'        # everything saved here becomes the notebook's output
!nvidia-smi
if not os.path.exists('/tmp/Pico'):
    !git clone https://github.com/Abhinav-Krishna-P/Pico-Next_Token_Predictor.git /tmp/Pico
%cd /tmp/Pico
!pip install -q sentencepiece datasets

## 2. Restore the previous run (if added as input)

In [ ]:
def find(pattern):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))
    return hits[0] if hits else None

for name in ['pico_big.pt', 'pico_big_tokenizer.model', 'pico_big_grammar.pt', 'pico_big_grammar_tokenizer.model']:
    src = find(name)
    if src and not os.path.exists(f'{W}/{name}'):
        shutil.copy(src, W)
        print('restored', name)

prev = find('data_big/train.bin')
if prev and not os.path.exists(f'{W}/data_big/train.bin'):
    shutil.copytree(os.path.dirname(prev), f'{W}/data_big')
    print('restored prepared data')

## 3. Prepare the data (first run only, ~30–60 min)
Full TinyStories + 300k FineWeb-Edu documents → 16k tokenizer → ~965M tokens.

In [ ]:
if not os.path.exists(f'{W}/data_big/train.bin'):
    !python prepare_data.py --out-dir {W}/data_big
    for f in glob.glob(f'{W}/data_big/raw_*.txt'):
        os.remove(f)     # raw text is no longer needed; keeps the output small
else:
    print('Data already prepared')

## 4. Pre-train (~34M parameters, 600M tokens)
Stops by itself before the 12-hour limit (keeping ~1 hour for fine-tuning), saving a checkpoint you can continue from.

In [ ]:
hours_left = 11.0 - (time.time() - START) / 3600
limit = max(0.25, hours_left - 1.0)
print(f'Pre-training time budget: {limit:.1f} hours')
!python pretrain.py --data-dir {W}/data_big --out {W}/pico_big.pt --time-limit-hours {limit:.2f}

## 5. Try the pre-trained model

In [ ]:
import torch, sentencepiece as spm
from model import Pico
ckpt = torch.load(f'{W}/pico_big.pt', map_location='cuda')
PRETRAIN_DONE = ckpt.get('finished', False)
sp = spm.SentencePieceProcessor(model_file=f'{W}/pico_big_tokenizer.model')
model = Pico(**ckpt['config']).cuda().eval()
model.load_state_dict(ckpt['model'])
print(f"step {ckpt['step']}, val loss {ckpt['val_loss']:.4f}, finished: {PRETRAIN_DONE}
")
for prompt in ['Once upon a time', 'The history of science', 'My favourite food is']:
    idx = torch.tensor([sp.encode(prompt)], device='cuda')
    print(sp.decode(model.generate(idx, max_new_tokens=100, stop_id=sp.eos_id())[0].tolist()), '
')
del model, ckpt
torch.cuda.empty_cache()

## 6. Fine-tune for grammar correction (~30 min)
Runs once pre-training has finished. Uses the `train.jsonl` from your input dataset.

In [ ]:
data = find('train.jsonl')
if not PRETRAIN_DONE:
    print('Pre-training not finished yet: add this notebook output as input and run again')
elif data is None:
    print('train.jsonl not found: add your dataset as input (Add Input)')
elif os.path.exists(f'{W}/pico_big_grammar.pt'):
    print('Grammar model already trained')
else:
    !python finetune_grammar.py --data {data} --base {W}/pico_big.pt         --out {W}/pico_big_grammar.pt --tokenizer-out {W}/pico_big_grammar_tokenizer.model --steps 6000

## 7. Test + export a small file for your laptop

In [ ]:
if os.path.exists(f'{W}/pico_big_grammar.pt'):
    !python export_model.py {W}/pico_big_grammar.pt {W}/pico_big_grammar_small.pt
    for s in ['they was playing in park', 'She go to school yesterday.', 'I has a apple.',
              'He dont like coffee.', 'I am work as software engineer.']:
        print('IN :', s)
        !python correct.py --model {W}/pico_big_grammar_small.pt "{s}"
        print()

## 8. Use it on your laptop

From this version's **Output** tab download **`pico_big_grammar_small.pt`** and **`pico_big_grammar_tokenizer.model`** into the Pico project folder, then:

```
python correct.py --model pico_big_grammar_small.pt "they was playing in park"
```